## ETL de el Set de Evaluación

In [1]:
import librosa
import numpy as np
import pandas as pd
import os
import warnings
import multiprocessing
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings('ignore', category=UserWarning, module='librosa')

# ── Rutas del eval set ──────────────────────────────────────────────
AUDIO_DIR_PATH  = '../data/LA/ASVspoof2019_LA_eval/flac/'
PROTOCOL_PATH   = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
OUTPUT_DIR      = '../Metricas/ETL_V2.1_eval/'

# ── Mismos hiperparámetros que el ETL de train/dev ──────────────────
N_FRAMES      = 5
FRAME_LENGTH  = 2048
TARGET_SR     = 16000
CORES         = multiprocessing.cpu_count()


### Funciones

In [2]:
def load_dataset_metadata(protocol_path):
    df = pd.read_csv(protocol_path, sep=' ', header=None,
                     names=['speaker', 'file_name', 'env', 'attack', 'label'])
    df['label'] = df['label'].map({'bonafide': 0, 'spoof': 1})
    return df[['file_name', 'label', 'attack']]

def load_and_preprocess_audio(file_path, target_sr=16000):
    audio, sr = librosa.load(file_path, sr=target_sr)
    return audio, sr

def calculate_uniform_segments(total_samples, n_frames=5, frame_length=2048):
    if total_samples < frame_length:
        raise ValueError(f"Audio demasiado corto: {total_samples} < {frame_length}")
    segment_size = total_samples // n_frames
    indices = []
    for i in range(n_frames):
        center = i * segment_size + segment_size // 2
        start  = max(0, center - frame_length // 2)
        end    = start + frame_length
        if end > total_samples:
            end   = total_samples
            start = end - frame_length
        indices.append((start, end))
    return indices

def extract_fourier_metrics(audio_frame, frame_length=2048):
    stft_matrix      = librosa.stft(audio_frame, n_fft=frame_length, hop_length=frame_length + 1)
    magnitude_spectrum = np.abs(stft_matrix)
    return magnitude_spectrum

def etl_uniform_extraction_pipeline(file_path, n_frames=5, frame_length=2048, target_sr=16000):
    audio, sr    = load_and_preprocess_audio(file_path, target_sr)
    frame_indices = calculate_uniform_segments(len(audio), n_frames, frame_length)
    features_matrix = []
    for start, end in frame_indices:
        frame   = audio[start:end]
        spectrum = extract_fourier_metrics(frame, frame_length)
        features_matrix.append(spectrum)
    return np.array(features_matrix)

def process_single_file_joblib(row, audio_dir, n_frames):
    file_path = os.path.join(audio_dir, f"{row['file_name']}.flac")
    try:
        features = etl_uniform_extraction_pipeline(file_path, n_frames=n_frames)
        return features, row['label']
    except Exception as e:
        return None, None

def run_joblib_etl(df, audio_dir, n_frames=5):
    rows    = [row for _, row in df.iterrows()]
    results = Parallel(n_jobs=CORES)(
        delayed(process_single_file_joblib)(row, audio_dir, n_frames)
        for row in tqdm(rows, desc='Extrayendo features eval')
    )
    X, y, errores = [], [], []
    for features, label in results:
        if features is not None:
            X.append(features)
            y.append(label)
        else:
            errores.append(1)
    print(f'Archivos procesados : {len(X)}')
    print(f'Errores             : {len(errores)}')
    return np.array(X), np.array(y), errores

def save_tensors_to_disk(X_tensor, y_tensor, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, 'X_fourier_features.npy'), X_tensor)
    np.save(os.path.join(output_dir, 'y_labels.npy'), y_tensor)
    print(f'X shape: {X_tensor.shape} → guardado en {output_dir}')
    print(f'y shape: {y_tensor.shape} → guardado en {output_dir}')


### Extracción

In [3]:
if __name__ == '__main__':
    df_eval = load_dataset_metadata(PROTOCOL_PATH)
    print(f'Archivos en eval set : {len(df_eval)}')
    print(f'Distribución labels  : {df_eval["label"].value_counts().to_dict()}')
    print(f'Tipos de spoof       : {sorted(df_eval["attack"].unique())}')

    X_eval, y_eval, errores = run_joblib_etl(df_eval, AUDIO_DIR_PATH, n_frames=N_FRAMES)
    save_tensors_to_disk(X_eval, y_eval, OUTPUT_DIR)


Archivos en eval set : 71237
Distribución labels  : {1: 63882, 0: 7355}
Tipos de spoof       : ['-', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19']


Extrayendo features eval: 100%|██████████| 71237/71237 [00:14<00:00, 4959.26it/s]


Archivos procesados : 71237
Errores             : 0
X shape: (71237, 5, 1025, 1) → guardado en ../Metricas/ETL_V2.1_eval/
y shape: (71237,) → guardado en ../Metricas/ETL_V2.1_eval/
